In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
import os
import json


In [4]:
combinedDatasets = []

In [5]:
dataset1 = pd.read_csv('/content/drive/MyDrive/Project_NLP/public_train (1).csv/public_train.csv')
publicTrainDataset = pd.DataFrame(dataset1)

for index, record in publicTrainDataset.iterrows():
    combinedDatasets.append({'text': record['post_message'], 'label': record['label']})


In [6]:
for root, dirs, files in os.walk('/content/drive/MyDrive/Project_NLP/Dataset/Misleading'):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                combinedDatasets.append({'text': data.get('text', data.get('maintext')), 'label': 0})


In [7]:
for root, dirs, files in os.walk('/content/drive/MyDrive/Project_NLP/Dataset/Fake'):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                combinedDatasets.append({'text': data.get('text', data.get('maintext')), 'label': 0})



In [8]:
for root, dirs, files in os.walk('/content/drive/MyDrive/Project_NLP/Dataset/Real'):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                combinedDatasets.append({'text': data.get('text', data.get('maintext')), 'label': 1})


In [9]:
for root, dirs, files in os.walk('/content/drive/MyDrive/Project_NLP/Fake_Real_Dataset/Fake'):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                combinedDatasets.append({'text': data.get('text', data.get('maintext')), 'label': 0})



In [10]:
for root, dirs, files in os.walk('/content/drive/MyDrive/Project_NLP/Fake_Real_Dataset/Real'):
    for file in files:
        if file.endswith('.json'):
            file_path = os.path.join(root, file)
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                combinedDatasets.append({'text': data.get('text', data.get('maintext')), 'label': 1})


In [11]:
# #Vietnamese Fake-News-Dataset-PBL7

updateDatasetPaths = os.listdir('/content/drive/MyDrive/Project_NLP/update_data')
for path in updateDatasetPaths:
    file = pd.read_csv('/content/drive/MyDrive/Project_NLP/update_data/'+path)
    for index, record in file.iterrows():
        combinedDatasets.append({'text': record['Maintext'], 'label': record['Label']})



In [12]:
pip install nltk

In [20]:
pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.5/897.5 kB 9.8 MB/s eta 0:00:00


In [30]:
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [29]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd
from tqdm import tqdm
import os

# Load mô hình VinAI
tokenizer = AutoTokenizer.from_pretrained("vinai/vinai-translate-en2vi-v2", src_lang="en_XX")
model = AutoModelForSeq2SeqLM.from_pretrained("vinai/vinai-translate-en2vi-v2")
device = torch.device("cuda")
model.to(device)

# Dịch một batch tiêu đề
def translate_batch(batch_titles, max_length=128):
    inputs = tokenizer(batch_titles, padding=True, truncation=True, max_length=max_length, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            decoder_start_token_id=tokenizer.lang_code_to_id["vi_VN"],
            num_beams=1
        )
    return tokenizer.batch_decode(outputs, skip_special_tokens=True)

# Thiết lập
batch_size = 64
save_every = 128  # lưu tạm mỗi 100 dòng
output_path = '/content/drive/MyDrive/Project_NLP/translated_titles_vinai.csv'

# Đọc dữ liệu (chỉnh cột nếu cần)
fake_df = pd.read_csv('/content/drive/MyDrive/Project_NLP/trueOrFake/Fake.csv')
true_df = pd.read_csv('/content/drive/MyDrive/Project_NLP/trueOrFake/True.csv')

# Kết hợp lại
fake_df['label'] = 0
true_df['label'] = 1
full_df = pd.concat([fake_df, true_df], ignore_index=True)
full_df = full_df[['title', 'label']].dropna(subset=['title']).reset_index(drop=True)

# Xác định số dòng đã dịch
if os.path.exists(output_path):
    done_df = pd.read_csv(output_path)
    start_index = len(done_df)
    translated_records = done_df.to_dict(orient='records')
    print(f"🔁 Tiếp tục từ dòng {start_index}")
else:
    start_index = 0
    translated_records = []
    print("🚀 Bắt đầu dịch từ đầu")

# Dịch phần còn lại
for i in tqdm(range(start_index, len(full_df), batch_size)):
    batch = full_df.iloc[i:i + batch_size]
    batch_titles = batch['title'].tolist()

    try:
        translated_texts = translate_batch(batch_titles)
    except Exception as e:
        print(f"⚠️ Lỗi ở batch {i}: {e}")
        translated_texts = ["LỖI DỊCH"] * len(batch)

    for text, label in zip(translated_texts, batch['label'].tolist()):
        translated_records.append({'text': text, 'label': label})

    # Lưu tạm
    if (i + batch_size) % save_every == 0 or (i + batch_size) >= len(full_df):
        pd.DataFrame(translated_records).to_csv(output_path, index=False)
        print(f"✅ Đã lưu tạm tại dòng {i + batch_size}")

print("🎉 Dịch hoàn tất hoặc đã dịch hết từ trước!")

RuntimeError: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx

In [1]:
print(combinedDatasets)

NameError: name 'combinedDatasets' is not defined

In [ ]:
combinedDatasets = pd.DataFrame(combinedDatasets)
combinedDatasets.to_csv("translated_news.csv", index=False, encoding='utf-8-sig')